# DS Quest blank-page practice: SQL for Data Science

**Blank-page version:** every code cell is empty - write the whole answer yourself. (The fill-in-the-blank version is `sql.ipynb`.)

Practical questions from the **SQL for Data Science** module. Write your answer in the code cell under each question, run it, then run the **check** cell right below it. The check runs *in this notebook*, on your own variables, and tells you exactly what is right or wrong.

1. Run the first code cell once (it loads the checker and the 12 questions).
2. Work through the questions in any order.
3. `dsq.hint("question-id")` gives a hint; `dsq.solution("question-id")` shows the reference answer (only after you tried 3 times).
4. `dsq.score()` shows where you stand.

Optional: `dsq.connect("http://127.0.0.1:5177", "YOUR-KEY")` (key from the module page in DS Quest) sends solved questions to the app so your profile counts them.

You need Python with `numpy`, `pandas`, `matplotlib` and `seaborn`, e.g. `pip install numpy pandas matplotlib seaborn notebook`.

In [ ]:
# --- DS Quest practice checker (runs in YOUR notebook, on YOUR variables) ---
import json as _json, sys as _sys, urllib.request as _ur

class _DSQ:
    def __init__(self):
        self.qs = {}; self.order = []; self.tries = {}; self.ok = {}; self.url = None; self.key = None; self.module = ''

    def load(self, pack):
        self.module = pack['title']
        for q in pack['questions']:
            self.qs[q['id']] = q; self.order.append(q['id'])
        print(f"DS Quest: {len(self.order)} questions loaded for '{self.module}'. Write your answer, then run the check cell under it.")
        if self._auto(): print('Linked to your DS Quest account: solved questions count in the app.')

    def connect(self, url, key, quiet=False):
        self.url = url.rstrip('/'); self.key = key
        if not quiet: print('Connected: solved questions will be reported to', self.url)

    def _auto(self):
        # DS Quest writes dsq_config.json next to the notebooks when you open them from the app
        import os
        d = os.getcwd()
        for _ in range(4):
            p = os.path.join(d, 'dsq_config.json')
            if os.path.exists(p):
                try:
                    c = _json.load(open(p)); self.connect(c['url'], c['key'], quiet=True); return True
                except Exception:
                    pass
            d = os.path.dirname(d)
        return False

    def _report(self, qid, ok):
        if not self.url: return
        try:
            req = _ur.Request(self.url + '/api/nb/report', data=_json.dumps({'key': self.key, 'qid': qid, 'ok': bool(ok)}).encode(), headers={'Content-Type': 'application/json'})
            _ur.urlopen(req, timeout=4).read()
        except Exception as e:
            print('(could not reach the app:', str(e)[:80] + ')')

    def check(self, qid):
        q = self.qs[qid]
        try:
            ip = get_ipython(); ns = dict(ip.user_ns)
        except NameError:
            ns = dict(_sys.modules['__main__'].__dict__)
        results = []
        def check(cond, msg, hint=''):
            ok = bool(cond); results.append((ok, msg, hint)); return ok
        ns['check'] = check; ns['OUT'] = ''
        for mod, alias in (('numpy', 'np'), ('pandas', 'pd')):
            if alias not in ns:
                try: ns[alias] = __import__(mod)
                except Exception: pass
        if 'plt' not in ns:
            try:
                import matplotlib.pyplot as plt; ns['plt'] = plt
            except Exception: pass
        try:
            exec(compile(q['tests'], '<tests>', 'exec'), ns)
        except BaseException as e:
            results.append((False, 'A check could not finish: ' + type(e).__name__ + ': ' + str(e)[:200], 'Did you create every variable name the task asks for? Did you run your answer cell first?'))
        self.tries[qid] = self.tries.get(qid, 0) + 1
        good = bool(results) and all(r[0] for r in results)
        for ok, msg, hint in results:
            print(('  [PASS] ' if ok else '  [FAIL] ') + msg + ('' if ok or not hint else '\n         hint: ' + hint))
        if good:
            self.ok[qid] = True; print(f"\n  Question solved: {q['title']}  (attempt {self.tries[qid]})")
        else:
            print(f"\n  Not yet - fix your answer cell, run it again, then re-run this check. (attempt {self.tries[qid]})")
        self._report(qid, good)
        return good

    def hint(self, qid):
        hs = self.qs[qid].get('hints') or []
        n = min(self.tries.get(qid, 0), len(hs)) if hs else 0
        print(('hint: ' + hs[max(0, n - 1)]) if hs else 'No hint for this one - re-read the question.')

    def solution(self, qid):
        if self.tries.get(qid, 0) < 3 and not self.ok.get(qid):
            print('Try the check at least 3 times first - the struggle is where the learning happens.'); return
        print(self.qs[qid]['solution'])

    def score(self):
        n = len(self.order); s = sum(1 for q in self.order if self.ok.get(q))
        print(f"Solved {s} of {n}.")
        for q in self.order:
            print(('  [x] ' if self.ok.get(q) else '  [ ] ') + q + ' - ' + self.qs[q]['title'] + ('' if self.ok.get(q) else f"  (tries: {self.tries.get(q, 0)})"))

dsq = _DSQ()


dsq.load({
 "module": "sql",
 "title": "SQL for Data Science",
 "questions": [
  {
   "id": "sq-1-1",
   "title": "Paid orders over 100, biggest first",
   "tests": "check(list(result.columns) == ['order_id', 'amount'], 'returns exactly the columns order_id and amount', 'SELECT order_id, amount')\ncheck(list(result.order_id) == [109, 103, 105, 101], 'orders 109, 103, 105, 101 in that order', \"WHERE status = 'paid' AND amount > 100 ORDER BY amount DESC\")",
   "hints": [
    "Two tests joined with AND.",
    "ORDER BY amount DESC puts the biggest on top."
   ],
   "solution": "import sqlite3\nimport pandas as pd\n\ncon = sqlite3.connect(\":memory:\")\ncon.executescript(\"\"\"\nCREATE TABLE customers (customer_id INTEGER PRIMARY KEY, name TEXT, city TEXT, signup_date TEXT);\nINSERT INTO customers VALUES\n (1,'Asha','Pune','2024-01-05'), (2,'Ben','Delhi','2024-01-20'), (3,'Chen','Pune','2024-02-11'),\n (4,'Dara','Mumbai','2024-02-14'), (5,'Eli','Delhi','2024-03-02'), (6,'Fay','Pune','2024-03-09');\nCREATE TABLE orders (order_id INTEGER PRIMARY KEY, customer_id INTEGER, order_date TEXT, status TEXT, amount REAL);\nINSERT INTO orders VALUES\n (101,1,'2024-02-01','paid',120), (102,1,'2024-02-15','paid',80), (103,2,'2024-02-03','paid',200),\n (104,2,'2024-03-01','refunded',60), (105,3,'2024-03-05','paid',150), (106,4,'2024-03-07','paid',90),\n (107,1,'2024-03-12','paid',40), (108,3,'2024-03-20','cancelled',75), (109,5,'2024-03-22','paid',300);\n\"\"\")\n\ndef q(sql):\n    return pd.read_sql(sql, con)\n\nresult = q(\"\"\"\nSELECT order_id, amount\nFROM orders\nWHERE status = 'paid' AND amount > 100\nORDER BY amount DESC\n\"\"\")\nprint(result)"
  },
  {
   "id": "sq-1-2",
   "title": "Customers in two cities",
   "tests": "check(list(result.columns) == ['name', 'city'], 'columns name and city', 'SELECT name, city')\ncheck(list(result.name) == ['Asha', 'Ben', 'Chen', 'Eli', 'Fay'], 'Asha, Ben, Chen, Eli, Fay by signup date', \"WHERE city IN ('Pune', 'Delhi') ORDER BY signup_date\")",
   "hints": [
    "IN ('Pune', 'Delhi') is shorthand for two ORs.",
    "Dates written as YYYY-MM-DD sort correctly as plain text."
   ],
   "solution": "import sqlite3\nimport pandas as pd\n\ncon = sqlite3.connect(\":memory:\")\ncon.executescript(\"\"\"\nCREATE TABLE customers (customer_id INTEGER PRIMARY KEY, name TEXT, city TEXT, signup_date TEXT);\nINSERT INTO customers VALUES\n (1,'Asha','Pune','2024-01-05'), (2,'Ben','Delhi','2024-01-20'), (3,'Chen','Pune','2024-02-11'),\n (4,'Dara','Mumbai','2024-02-14'), (5,'Eli','Delhi','2024-03-02'), (6,'Fay','Pune','2024-03-09');\nCREATE TABLE orders (order_id INTEGER PRIMARY KEY, customer_id INTEGER, order_date TEXT, status TEXT, amount REAL);\nINSERT INTO orders VALUES\n (101,1,'2024-02-01','paid',120), (102,1,'2024-02-15','paid',80), (103,2,'2024-02-03','paid',200),\n (104,2,'2024-03-01','refunded',60), (105,3,'2024-03-05','paid',150), (106,4,'2024-03-07','paid',90),\n (107,1,'2024-03-12','paid',40), (108,3,'2024-03-20','cancelled',75), (109,5,'2024-03-22','paid',300);\n\"\"\")\n\ndef q(sql):\n    return pd.read_sql(sql, con)\n\nresult = q(\"\"\"\nSELECT name, city\nFROM customers\nWHERE city IN ('Pune', 'Delhi')\nORDER BY signup_date\n\"\"\")\nprint(result)"
  },
  {
   "id": "sq-1-3",
   "title": "Names that contain an e",
   "tests": "check(list(result.columns) == ['name'], 'one column: name', 'SELECT name')\ncheck(list(result.name) == ['Ben', 'Chen', 'Eli'], 'Ben, Chen, Eli', \"WHERE name LIKE '%e%' ORDER BY name\")",
   "hints": [
    "LIKE '%e%': the % signs mean 'anything before and after'.",
    "SQLite LIKE ignores upper and lower case for plain letters."
   ],
   "solution": "import sqlite3\nimport pandas as pd\n\ncon = sqlite3.connect(\":memory:\")\ncon.executescript(\"\"\"\nCREATE TABLE customers (customer_id INTEGER PRIMARY KEY, name TEXT, city TEXT, signup_date TEXT);\nINSERT INTO customers VALUES\n (1,'Asha','Pune','2024-01-05'), (2,'Ben','Delhi','2024-01-20'), (3,'Chen','Pune','2024-02-11'),\n (4,'Dara','Mumbai','2024-02-14'), (5,'Eli','Delhi','2024-03-02'), (6,'Fay','Pune','2024-03-09');\nCREATE TABLE orders (order_id INTEGER PRIMARY KEY, customer_id INTEGER, order_date TEXT, status TEXT, amount REAL);\nINSERT INTO orders VALUES\n (101,1,'2024-02-01','paid',120), (102,1,'2024-02-15','paid',80), (103,2,'2024-02-03','paid',200),\n (104,2,'2024-03-01','refunded',60), (105,3,'2024-03-05','paid',150), (106,4,'2024-03-07','paid',90),\n (107,1,'2024-03-12','paid',40), (108,3,'2024-03-20','cancelled',75), (109,5,'2024-03-22','paid',300);\n\"\"\")\n\ndef q(sql):\n    return pd.read_sql(sql, con)\n\nresult = q(\"\"\"\nSELECT name\nFROM customers\nWHERE name LIKE '%e%'\nORDER BY name\n\"\"\")\nprint(result)"
  },
  {
   "id": "sq-2-1",
   "title": "Revenue per customer",
   "tests": "check(list(result.columns) == ['customer_id', 'n_orders', 'revenue'], 'columns customer_id, n_orders, revenue', 'use AS to name the two aggregates')\ncheck(list(result.customer_id) == [5, 1, 2, 3, 4], 'customers 5, 1, 2, 3, 4 by revenue', 'GROUP BY customer_id ORDER BY revenue DESC')\ncheck(list(result.revenue) == [300, 240, 200, 150, 90], 'revenues 300, 240, 200, 150, 90', \"only paid orders: WHERE status = 'paid'\")\ncheck(int(result.set_index('customer_id').loc[1, 'n_orders']) == 3, 'customer 1 has 3 paid orders', 'COUNT(*)')",
   "hints": [
    "Filter with WHERE first, then GROUP BY customer_id.",
    "Name the two aggregates with AS n_orders and AS revenue."
   ],
   "solution": "import sqlite3\nimport pandas as pd\n\ncon = sqlite3.connect(\":memory:\")\ncon.executescript(\"\"\"\nCREATE TABLE customers (customer_id INTEGER PRIMARY KEY, name TEXT, city TEXT, signup_date TEXT);\nINSERT INTO customers VALUES\n (1,'Asha','Pune','2024-01-05'), (2,'Ben','Delhi','2024-01-20'), (3,'Chen','Pune','2024-02-11'),\n (4,'Dara','Mumbai','2024-02-14'), (5,'Eli','Delhi','2024-03-02'), (6,'Fay','Pune','2024-03-09');\nCREATE TABLE orders (order_id INTEGER PRIMARY KEY, customer_id INTEGER, order_date TEXT, status TEXT, amount REAL);\nINSERT INTO orders VALUES\n (101,1,'2024-02-01','paid',120), (102,1,'2024-02-15','paid',80), (103,2,'2024-02-03','paid',200),\n (104,2,'2024-03-01','refunded',60), (105,3,'2024-03-05','paid',150), (106,4,'2024-03-07','paid',90),\n (107,1,'2024-03-12','paid',40), (108,3,'2024-03-20','cancelled',75), (109,5,'2024-03-22','paid',300);\n\"\"\")\n\ndef q(sql):\n    return pd.read_sql(sql, con)\n\nresult = q(\"\"\"\nSELECT customer_id, COUNT(*) AS n_orders, SUM(amount) AS revenue\nFROM orders\nWHERE status = 'paid'\nGROUP BY customer_id\nORDER BY revenue DESC\n\"\"\")\nprint(result)"
  },
  {
   "id": "sq-2-2",
   "title": "Repeat customers with HAVING",
   "tests": "check(list(result.columns) == ['customer_id', 'n'], 'columns customer_id and n', 'SELECT customer_id, COUNT(*) AS n')\ncheck(list(result.customer_id) == [1, 2, 3], 'customers 1, 2 and 3', 'HAVING COUNT(*) >= 2')\ncheck(list(result.n) == [3, 2, 2], 'with 3, 2 and 2 orders', 'COUNT(*) counts all orders of the customer')",
   "hints": [
    "HAVING goes after GROUP BY.",
    "HAVING COUNT(*) >= 2"
   ],
   "solution": "import sqlite3\nimport pandas as pd\n\ncon = sqlite3.connect(\":memory:\")\ncon.executescript(\"\"\"\nCREATE TABLE customers (customer_id INTEGER PRIMARY KEY, name TEXT, city TEXT, signup_date TEXT);\nINSERT INTO customers VALUES\n (1,'Asha','Pune','2024-01-05'), (2,'Ben','Delhi','2024-01-20'), (3,'Chen','Pune','2024-02-11'),\n (4,'Dara','Mumbai','2024-02-14'), (5,'Eli','Delhi','2024-03-02'), (6,'Fay','Pune','2024-03-09');\nCREATE TABLE orders (order_id INTEGER PRIMARY KEY, customer_id INTEGER, order_date TEXT, status TEXT, amount REAL);\nINSERT INTO orders VALUES\n (101,1,'2024-02-01','paid',120), (102,1,'2024-02-15','paid',80), (103,2,'2024-02-03','paid',200),\n (104,2,'2024-03-01','refunded',60), (105,3,'2024-03-05','paid',150), (106,4,'2024-03-07','paid',90),\n (107,1,'2024-03-12','paid',40), (108,3,'2024-03-20','cancelled',75), (109,5,'2024-03-22','paid',300);\n\"\"\")\n\ndef q(sql):\n    return pd.read_sql(sql, con)\n\nresult = q(\"\"\"\nSELECT customer_id, COUNT(*) AS n\nFROM orders\nGROUP BY customer_id\nHAVING COUNT(*) >= 2\nORDER BY customer_id\n\"\"\")\nprint(result)"
  },
  {
   "id": "sq-2-3",
   "title": "Monthly paid revenue",
   "tests": "check(list(result.columns) == ['month', 'revenue'], 'columns month and revenue', \"SELECT strftime('%Y-%m', order_date) AS month, SUM(amount) AS revenue\")\ncheck(list(result.month) == ['2024-02', '2024-03'], 'February then March 2024', 'GROUP BY month ORDER BY month')\ncheck(list(result.revenue) == [400, 580], 'revenue 400 and 580', \"only paid orders: WHERE status = 'paid'\")",
   "hints": [
    "strftime('%Y-%m', order_date) AS month",
    "GROUP BY the alias, then ORDER BY it."
   ],
   "solution": "import sqlite3\nimport pandas as pd\n\ncon = sqlite3.connect(\":memory:\")\ncon.executescript(\"\"\"\nCREATE TABLE customers (customer_id INTEGER PRIMARY KEY, name TEXT, city TEXT, signup_date TEXT);\nINSERT INTO customers VALUES\n (1,'Asha','Pune','2024-01-05'), (2,'Ben','Delhi','2024-01-20'), (3,'Chen','Pune','2024-02-11'),\n (4,'Dara','Mumbai','2024-02-14'), (5,'Eli','Delhi','2024-03-02'), (6,'Fay','Pune','2024-03-09');\nCREATE TABLE orders (order_id INTEGER PRIMARY KEY, customer_id INTEGER, order_date TEXT, status TEXT, amount REAL);\nINSERT INTO orders VALUES\n (101,1,'2024-02-01','paid',120), (102,1,'2024-02-15','paid',80), (103,2,'2024-02-03','paid',200),\n (104,2,'2024-03-01','refunded',60), (105,3,'2024-03-05','paid',150), (106,4,'2024-03-07','paid',90),\n (107,1,'2024-03-12','paid',40), (108,3,'2024-03-20','cancelled',75), (109,5,'2024-03-22','paid',300);\n\"\"\")\n\ndef q(sql):\n    return pd.read_sql(sql, con)\n\nresult = q(\"\"\"\nSELECT strftime('%Y-%m', order_date) AS month, SUM(amount) AS revenue\nFROM orders\nWHERE status = 'paid'\nGROUP BY month\nORDER BY month\n\"\"\")\nprint(result)"
  },
  {
   "id": "sq-3-1",
   "title": "Paid orders with the customer name",
   "tests": "check(list(result.columns) == ['order_id', 'name', 'city', 'amount'], 'columns order_id, name, city, amount', 'SELECT o.order_id, c.name, c.city, o.amount')\ncheck(list(result.order_id) == [101, 102, 103, 105, 106, 107, 109], 'the seven paid orders', \"WHERE o.status = 'paid'\")\ncheck(list(result.name) == ['Asha', 'Asha', 'Ben', 'Chen', 'Dara', 'Asha', 'Eli'], 'each with the right customer name', 'JOIN on customer_id')",
   "hints": [
    "JOIN customers AS c ON c.customer_id = o.customer_id",
    "Filter with WHERE o.status = 'paid'."
   ],
   "solution": "import sqlite3\nimport pandas as pd\n\ncon = sqlite3.connect(\":memory:\")\ncon.executescript(\"\"\"\nCREATE TABLE customers (customer_id INTEGER PRIMARY KEY, name TEXT, city TEXT, signup_date TEXT);\nINSERT INTO customers VALUES\n (1,'Asha','Pune','2024-01-05'), (2,'Ben','Delhi','2024-01-20'), (3,'Chen','Pune','2024-02-11'),\n (4,'Dara','Mumbai','2024-02-14'), (5,'Eli','Delhi','2024-03-02'), (6,'Fay','Pune','2024-03-09');\nCREATE TABLE orders (order_id INTEGER PRIMARY KEY, customer_id INTEGER, order_date TEXT, status TEXT, amount REAL);\nINSERT INTO orders VALUES\n (101,1,'2024-02-01','paid',120), (102,1,'2024-02-15','paid',80), (103,2,'2024-02-03','paid',200),\n (104,2,'2024-03-01','refunded',60), (105,3,'2024-03-05','paid',150), (106,4,'2024-03-07','paid',90),\n (107,1,'2024-03-12','paid',40), (108,3,'2024-03-20','cancelled',75), (109,5,'2024-03-22','paid',300);\n\"\"\")\n\ndef q(sql):\n    return pd.read_sql(sql, con)\n\nresult = q(\"\"\"\nSELECT o.order_id, c.name, c.city, o.amount\nFROM orders AS o\nJOIN customers AS c ON c.customer_id = o.customer_id\nWHERE o.status = 'paid'\nORDER BY o.order_id\n\"\"\")\nprint(result)"
  },
  {
   "id": "sq-3-2",
   "title": "Customers who never ordered",
   "tests": "check(list(result.columns) == ['name'], 'one column: name', 'SELECT c.name')\ncheck(list(result.name) == ['Fay'], 'only Fay never ordered', 'LEFT JOIN ... WHERE o.order_id IS NULL')",
   "hints": [
    "Start FROM customers so none are lost.",
    "WHERE o.order_id IS NULL keeps only the unmatched ones."
   ],
   "solution": "import sqlite3\nimport pandas as pd\n\ncon = sqlite3.connect(\":memory:\")\ncon.executescript(\"\"\"\nCREATE TABLE customers (customer_id INTEGER PRIMARY KEY, name TEXT, city TEXT, signup_date TEXT);\nINSERT INTO customers VALUES\n (1,'Asha','Pune','2024-01-05'), (2,'Ben','Delhi','2024-01-20'), (3,'Chen','Pune','2024-02-11'),\n (4,'Dara','Mumbai','2024-02-14'), (5,'Eli','Delhi','2024-03-02'), (6,'Fay','Pune','2024-03-09');\nCREATE TABLE orders (order_id INTEGER PRIMARY KEY, customer_id INTEGER, order_date TEXT, status TEXT, amount REAL);\nINSERT INTO orders VALUES\n (101,1,'2024-02-01','paid',120), (102,1,'2024-02-15','paid',80), (103,2,'2024-02-03','paid',200),\n (104,2,'2024-03-01','refunded',60), (105,3,'2024-03-05','paid',150), (106,4,'2024-03-07','paid',90),\n (107,1,'2024-03-12','paid',40), (108,3,'2024-03-20','cancelled',75), (109,5,'2024-03-22','paid',300);\n\"\"\")\n\ndef q(sql):\n    return pd.read_sql(sql, con)\n\nresult = q(\"\"\"\nSELECT c.name\nFROM customers AS c\nLEFT JOIN orders AS o ON o.customer_id = c.customer_id\nWHERE o.order_id IS NULL\n\"\"\")\nprint(result)"
  },
  {
   "id": "sq-3-3",
   "title": "Paid revenue by city",
   "tests": "check(list(result.columns) == ['city', 'revenue'], 'columns city and revenue', 'SELECT c.city, SUM(o.amount) AS revenue')\ncheck(list(result.city) == ['Delhi', 'Pune', 'Mumbai'], 'Delhi, Pune, Mumbai by revenue', 'GROUP BY c.city ORDER BY revenue DESC')\ncheck(list(result.revenue) == [500, 390, 90], 'revenue 500, 390, 90', \"only paid orders: WHERE o.status = 'paid'\")",
   "hints": [
    "Join first, then GROUP BY c.city.",
    "Delhi: Ben 200 + Eli 300."
   ],
   "solution": "import sqlite3\nimport pandas as pd\n\ncon = sqlite3.connect(\":memory:\")\ncon.executescript(\"\"\"\nCREATE TABLE customers (customer_id INTEGER PRIMARY KEY, name TEXT, city TEXT, signup_date TEXT);\nINSERT INTO customers VALUES\n (1,'Asha','Pune','2024-01-05'), (2,'Ben','Delhi','2024-01-20'), (3,'Chen','Pune','2024-02-11'),\n (4,'Dara','Mumbai','2024-02-14'), (5,'Eli','Delhi','2024-03-02'), (6,'Fay','Pune','2024-03-09');\nCREATE TABLE orders (order_id INTEGER PRIMARY KEY, customer_id INTEGER, order_date TEXT, status TEXT, amount REAL);\nINSERT INTO orders VALUES\n (101,1,'2024-02-01','paid',120), (102,1,'2024-02-15','paid',80), (103,2,'2024-02-03','paid',200),\n (104,2,'2024-03-01','refunded',60), (105,3,'2024-03-05','paid',150), (106,4,'2024-03-07','paid',90),\n (107,1,'2024-03-12','paid',40), (108,3,'2024-03-20','cancelled',75), (109,5,'2024-03-22','paid',300);\n\"\"\")\n\ndef q(sql):\n    return pd.read_sql(sql, con)\n\nresult = q(\"\"\"\nSELECT c.city, SUM(o.amount) AS revenue\nFROM orders AS o\nJOIN customers AS c ON c.customer_id = o.customer_id\nWHERE o.status = 'paid'\nGROUP BY c.city\nORDER BY revenue DESC\n\"\"\")\nprint(result)"
  },
  {
   "id": "sq-4-1",
   "title": "Customers above the average (CTE)",
   "tests": "check(list(result.columns) == ['customer_id', 'revenue'], 'columns customer_id and revenue', 'SELECT customer_id, revenue FROM rev')\ncheck(list(result.customer_id) == [5, 1, 2], 'customers 5, 1, 2 beat the average of 196', 'WHERE revenue > (SELECT AVG(revenue) FROM rev)')\ncheck(list(result.revenue) == [300, 240, 200], 'revenue 300, 240, 200', 'ORDER BY revenue DESC')",
   "hints": [
    "The CTE is the same query as the earlier revenue-per-customer one.",
    "(SELECT AVG(revenue) FROM rev) is a subquery that gives one number: 196."
   ],
   "solution": "import sqlite3\nimport pandas as pd\n\ncon = sqlite3.connect(\":memory:\")\ncon.executescript(\"\"\"\nCREATE TABLE customers (customer_id INTEGER PRIMARY KEY, name TEXT, city TEXT, signup_date TEXT);\nINSERT INTO customers VALUES\n (1,'Asha','Pune','2024-01-05'), (2,'Ben','Delhi','2024-01-20'), (3,'Chen','Pune','2024-02-11'),\n (4,'Dara','Mumbai','2024-02-14'), (5,'Eli','Delhi','2024-03-02'), (6,'Fay','Pune','2024-03-09');\nCREATE TABLE orders (order_id INTEGER PRIMARY KEY, customer_id INTEGER, order_date TEXT, status TEXT, amount REAL);\nINSERT INTO orders VALUES\n (101,1,'2024-02-01','paid',120), (102,1,'2024-02-15','paid',80), (103,2,'2024-02-03','paid',200),\n (104,2,'2024-03-01','refunded',60), (105,3,'2024-03-05','paid',150), (106,4,'2024-03-07','paid',90),\n (107,1,'2024-03-12','paid',40), (108,3,'2024-03-20','cancelled',75), (109,5,'2024-03-22','paid',300);\n\"\"\")\n\ndef q(sql):\n    return pd.read_sql(sql, con)\n\nresult = q(\"\"\"\nWITH rev AS (\n  SELECT customer_id, SUM(amount) AS revenue\n  FROM orders\n  WHERE status = 'paid'\n  GROUP BY customer_id\n)\nSELECT customer_id, revenue\nFROM rev\nWHERE revenue > (SELECT AVG(revenue) FROM rev)\nORDER BY revenue DESC\n\"\"\")\nprint(result)"
  },
  {
   "id": "sq-4-2",
   "title": "Latest order per customer (ROW_NUMBER)",
   "tests": "check(list(result.columns) == ['customer_id', 'order_id', 'order_date'], 'columns customer_id, order_id, order_date', 'SELECT customer_id, order_id, order_date FROM (...)')\ncheck(list(result.order_id) == [107, 104, 108, 106, 109], 'latest orders 107, 104, 108, 106, 109', 'PARTITION BY customer_id ORDER BY order_date DESC, then rn = 1')",
   "hints": [
    "ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY order_date DESC)",
    "Wrap that query in parentheses after FROM and filter WHERE rn = 1."
   ],
   "solution": "import sqlite3\nimport pandas as pd\n\ncon = sqlite3.connect(\":memory:\")\ncon.executescript(\"\"\"\nCREATE TABLE customers (customer_id INTEGER PRIMARY KEY, name TEXT, city TEXT, signup_date TEXT);\nINSERT INTO customers VALUES\n (1,'Asha','Pune','2024-01-05'), (2,'Ben','Delhi','2024-01-20'), (3,'Chen','Pune','2024-02-11'),\n (4,'Dara','Mumbai','2024-02-14'), (5,'Eli','Delhi','2024-03-02'), (6,'Fay','Pune','2024-03-09');\nCREATE TABLE orders (order_id INTEGER PRIMARY KEY, customer_id INTEGER, order_date TEXT, status TEXT, amount REAL);\nINSERT INTO orders VALUES\n (101,1,'2024-02-01','paid',120), (102,1,'2024-02-15','paid',80), (103,2,'2024-02-03','paid',200),\n (104,2,'2024-03-01','refunded',60), (105,3,'2024-03-05','paid',150), (106,4,'2024-03-07','paid',90),\n (107,1,'2024-03-12','paid',40), (108,3,'2024-03-20','cancelled',75), (109,5,'2024-03-22','paid',300);\n\"\"\")\n\ndef q(sql):\n    return pd.read_sql(sql, con)\n\nresult = q(\"\"\"\nSELECT customer_id, order_id, order_date\nFROM (\n  SELECT *, ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY order_date DESC, order_id DESC) AS rn\n  FROM orders\n)\nWHERE rn = 1\nORDER BY customer_id\n\"\"\")\nprint(result)"
  },
  {
   "id": "sq-4-3",
   "title": "A running total of paid revenue",
   "tests": "check(list(result.columns) == ['order_id', 'order_date', 'amount', 'running'], 'columns order_id, order_date, amount, running', 'name the window column AS running')\ncheck(list(result.order_id) == [101, 103, 102, 105, 106, 107, 109], 'paid orders in date order', 'ORDER BY order_date, order_id')\ncheck(list(result.running) == [120, 320, 400, 550, 640, 680, 980], 'running total 120, 320, 400, 550, 640, 680, 980', 'SUM(amount) OVER (ORDER BY order_date, order_id)')",
   "hints": [
    "SUM(amount) OVER (ORDER BY order_date, order_id)",
    "WHERE removes the unpaid rows before the window is computed."
   ],
   "solution": "import sqlite3\nimport pandas as pd\n\ncon = sqlite3.connect(\":memory:\")\ncon.executescript(\"\"\"\nCREATE TABLE customers (customer_id INTEGER PRIMARY KEY, name TEXT, city TEXT, signup_date TEXT);\nINSERT INTO customers VALUES\n (1,'Asha','Pune','2024-01-05'), (2,'Ben','Delhi','2024-01-20'), (3,'Chen','Pune','2024-02-11'),\n (4,'Dara','Mumbai','2024-02-14'), (5,'Eli','Delhi','2024-03-02'), (6,'Fay','Pune','2024-03-09');\nCREATE TABLE orders (order_id INTEGER PRIMARY KEY, customer_id INTEGER, order_date TEXT, status TEXT, amount REAL);\nINSERT INTO orders VALUES\n (101,1,'2024-02-01','paid',120), (102,1,'2024-02-15','paid',80), (103,2,'2024-02-03','paid',200),\n (104,2,'2024-03-01','refunded',60), (105,3,'2024-03-05','paid',150), (106,4,'2024-03-07','paid',90),\n (107,1,'2024-03-12','paid',40), (108,3,'2024-03-20','cancelled',75), (109,5,'2024-03-22','paid',300);\n\"\"\")\n\ndef q(sql):\n    return pd.read_sql(sql, con)\n\nresult = q(\"\"\"\nSELECT order_id, order_date, amount,\n       SUM(amount) OVER (ORDER BY order_date, order_id) AS running\nFROM orders\nWHERE status = 'paid'\nORDER BY order_date, order_id\n\"\"\")\nprint(result)"
  }
 ]
})


## 1. Paid orders over 100, biggest first  `sq-1-1`
*(from the lesson: SELECT: asking a database a question)*

Write a query that returns `order_id` and `amount` of every order that is **paid** and has an amount **above 100**, biggest amount first. Keep it in `result`.

The helper `q("...")` runs your SQL on the shop database and returns a DataFrame.

In [ ]:
# your code here


In [ ]:
dsq.check("sq-1-1")

## 2. Customers in two cities  `sq-1-2`
*(from the lesson: SELECT: asking a database a question)*

Return the `name` and `city` of customers who live in **Pune or Delhi**, ordered by `signup_date` (oldest first).

In [ ]:
# your code here


In [ ]:
dsq.check("sq-1-2")

## 3. Names that contain an e  `sq-1-3`
*(from the lesson: SELECT: asking a database a question)*

Return only the `name` column for customers whose name **contains the letter e** (any position), in alphabetical order.

In [ ]:
# your code here


In [ ]:
dsq.check("sq-1-3")

## 4. Revenue per customer  `sq-2-1`
*(from the lesson: GROUP BY: from rows to answers)*

For **paid** orders only, return one row per customer: `customer_id`, `n_orders` (how many paid orders) and `revenue` (their sum), with the highest revenue first.

In [ ]:
# your code here


In [ ]:
dsq.check("sq-2-1")

## 5. Repeat customers with HAVING  `sq-2-2`
*(from the lesson: GROUP BY: from rows to answers)*

Which customers placed **at least 2 orders** of any status? Return `customer_id` and `n` (the number of orders), ordered by `customer_id`.

You cannot use `WHERE COUNT(*) >= 2`: the count only exists after grouping.

In [ ]:
# your code here


In [ ]:
dsq.check("sq-2-2")

## 6. Monthly paid revenue  `sq-2-3`
*(from the lesson: GROUP BY: from rows to answers)*

Return paid revenue per **month**: columns `month` (like `2024-02`) and `revenue`, oldest month first. Use `strftime('%Y-%m', order_date)`.

In [ ]:
# your code here


In [ ]:
dsq.check("sq-2-3")

## 7. Paid orders with the customer name  `sq-3-1`
*(from the lesson: JOIN: putting tables together)*

Return `order_id`, `name`, `city` and `amount` for every **paid** order, joining `orders` to `customers`. Order by `order_id`.

In [ ]:
# your code here


In [ ]:
dsq.check("sq-3-1")

## 8. Customers who never ordered  `sq-3-2`
*(from the lesson: JOIN: putting tables together)*

Which customers have **no orders at all**? Return their `name`. Use a `LEFT JOIN` and keep the rows where the order side is `NULL`.

In [ ]:
# your code here


In [ ]:
dsq.check("sq-3-2")

## 9. Paid revenue by city  `sq-3-3`
*(from the lesson: JOIN: putting tables together)*

The city lives in `customers`, the money in `orders`. Return `city` and `revenue` (sum of **paid** amounts), highest revenue first.

In [ ]:
# your code here


In [ ]:
dsq.check("sq-3-3")

## 10. Customers above the average (CTE)  `sq-4-1`
*(from the lesson: CTEs and window functions)*

First build each customer's paid revenue in a CTE. Then return the customers whose revenue is **above the average customer revenue**: columns `customer_id`, `revenue`, highest first.

In [ ]:
# your code here


In [ ]:
dsq.check("sq-4-1")

## 11. Latest order per customer (ROW_NUMBER)  `sq-4-2`
*(from the lesson: CTEs and window functions)*

Return each customer's **most recent order** (any status): `customer_id`, `order_id`, `order_date`, ordered by `customer_id`.

Number each customer's orders newest-first with `ROW_NUMBER()`, then keep number 1 (do the filtering in an outer query).

In [ ]:
# your code here


In [ ]:
dsq.check("sq-4-2")

## 12. A running total of paid revenue  `sq-4-3`
*(from the lesson: CTEs and window functions)*

For **paid** orders, return `order_id`, `order_date`, `amount` and `running`: the total of paid amounts so far when the orders are taken in date order (break date ties by `order_id`). Output in the same order.

In [ ]:
# your code here


In [ ]:
dsq.check("sq-4-3")

## Done?

In [ ]:
dsq.score()